# gas-sim-pro · Training Notebook
Run all cells top to bottom. Do not skip cells.
Training completes in ~10 minutes on Colab free tier.

In [ ]:
# ── Cell 1 — Config + registry check ─────────────────────────────────────
PROJECT_ID = 'gas-sim-pro-ii'   # ← your project ID
BUCKET     = f'{PROJECT_ID}-gas-sim-data'

from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
import json

gcs    = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET)
reg    = json.loads(bucket.blob('model_registry.json').download_as_text())

print('── Registry ──────────────────────────────')
print(f'  last_trained:     {reg["last_trained"]}')
print(f'  last_data_upload: {reg["last_data_upload"]}')
print(f'  feature_version:  {reg["feature_version"]}')
print(f'  current MAE:      {reg["mae"]}')
print(f'  rows_trained_on:  {reg["rows_trained_on"]}')
print('──────────────────────────────────────────')
print('Registry loaded. Proceed to Cell 2.')

In [ ]:
# ── Cell 2 — Install dependencies ────────────────────────────────────────
!pip install -q xgboost wandb onnx onnxmltools skl2onnx pyarrow pandas-gbq

In [ ]:
# ── Cell 3 — Load features from GCS ──────────────────────────────────────
import pandas as pd

# Find all parquet files for this feature version
blobs = list(gcs.bucket(BUCKET).list_blobs(prefix='features/latest/'))
paths = [f'gs://{BUCKET}/{b.name}' for b in blobs if b.name.endswith('.parquet')]
print(f'Found {len(paths)} parquet file(s)')

df = pd.concat(
    [pd.read_parquet(p, storage_options={'token': 'google_default'}) for p in paths],
    ignore_index=True
)

assert len(df) >= 500, f'Only {len(df)} rows — need at least 500 to train'
print(f'Loaded {len(df):,} rows · {len(df.columns)} columns')
df[['sensor_delta','centroid_row','centroid_col','wind_angle','leak_row','leak_col']].describe()

In [ ]:
# ── Cell 4 — Feature matrix + train/val split ─────────────────────────────
from sklearn.model_selection import train_test_split
import numpy as np

FEATURES = [
    'sensor_delta', 'sensor_mean', 'reading_variance',
    'centroid_row', 'centroid_col', 'coverage_ratio',
    'wind_angle', 'wind_magnitude', 'distance_to_boundary',
    'wind_x', 'wind_y', 'diffusion_rate', 'decay_factor',
    'leak_injection', 'sensor_count'
]
TARGETS = ['leak_row', 'leak_col']

X = df[FEATURES].values.astype(np.float32)
y = df[TARGETS].values.astype(np.float32)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=42
)
print(f'Train: {len(X_train):,}  Val: {len(X_val):,}')

In [ ]:
# ── Cell 5 — W&B init ─────────────────────────────────────────────────────
import wandb

run = wandb.init(
    project='gas-sim-pro',
    config={
        'model': 'xgboost',
        'features': FEATURES,
        'n_rows': len(df),
        'feature_version': reg['feature_version'],
    }
)
print(f'W&B run: {run.url}')

In [ ]:
# ── Cell 6 — Train XGBoost ────────────────────────────────────────────────
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor

params = dict(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0
)
model = MultiOutputRegressor(xgb.XGBRegressor(**params))
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
mae    = float(np.mean(np.abs(y_pred - y_val)))
print(f'Val MAE: {mae:.3f} cells')
wandb.log({'val_mae': mae})

In [ ]:
# ── Cell 7 — MAE regression gate ─────────────────────────────────────────
prod_mae = reg.get('mae') or float('inf')

if mae >= prod_mae * 0.98:
    print(f'Gate FAILED: new MAE {mae:.3f} does not beat prod {prod_mae:.3f} by >2%')
    wandb.finish(exit_code=1)
    raise SystemExit('MAE gate failed — no model exported')

print(f'Gate PASSED: {mae:.3f} < {prod_mae:.3f} * 0.98')

In [ ]:
# ── Cell 8 — ONNX export ──────────────────────────────────────────────────
import os, datetime, joblib
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

RUN_ID  = run.id
VERSION = f"v{datetime.date.today().strftime('%Y%m%d')}-{RUN_ID}"
os.makedirs(f'/tmp/{VERSION}', exist_ok=True)

# joblib
joblib.dump(model, f'/tmp/{VERSION}/model.joblib')

# ONNX
initial_type = [('float_input', FloatTensorType([None, len(FEATURES)]))]
onnx_model   = convert_sklearn(model, initial_types=initial_type)
with open(f'/tmp/{VERSION}/model.onnx', 'wb') as f:
    f.write(onnx_model.SerializeToString())

print(f'Exported: {VERSION}')

In [ ]:
# ── Cell 9 — Push to GCS ──────────────────────────────────────────────────
for fname in ['model.joblib', 'model.onnx']:
    blob = bucket.blob(f'models/{VERSION}/{fname}')
    blob.upload_from_filename(f'/tmp/{VERSION}/{fname}')
    print(f'Uploaded: models/{VERSION}/{fname}')

wandb.log_artifact(f'/tmp/{VERSION}/model.onnx', name='model-onnx', type='model')

In [ ]:
# ── Cell 10 — Update model_registry.json ─────────────────────────────────
import datetime as dt

prev_version = reg.get('latest_version')
prev_mae     = reg.get('mae')

reg.update({
    'latest_version':   VERSION,
    'previous_version': prev_version,
    'last_trained':     dt.datetime.now(dt.timezone.utc).isoformat(),
    'model_path':       f'models/{VERSION}/model.onnx',
    'joblib_path':      f'models/{VERSION}/model.joblib',
    'mae':              mae,
    'previous_mae':     prev_mae,
    'rows_trained_on':  len(df),
    'feature_version':  reg['feature_version'],
    'gate_status':      'passed',
})

reg_blob = bucket.blob('model_registry.json')
reg_blob.upload_from_string(json.dumps(reg, indent=2), content_type='application/json')
reg_blob.cache_control = 'no-cache, no-store, max-age=0'
reg_blob.patch()

print(f'Registry updated → {VERSION}')
print(f'MAE: {mae:.3f}  (prev: {prev_mae})')
wandb.finish()
print('Training complete. ENV D (Cloud Run deployment) will trigger automatically.')

In [ ]:
# ── Cell 11 — Verify ONNX output ─────────────────────────────────────────
import onnxruntime as rt

sess      = rt.InferenceSession(f'/tmp/{VERSION}/model.onnx')
test_in   = X_val[:5]
onnx_pred = sess.run(None, {'float_input': test_in})
skl_pred  = model.predict(test_in)

print('ONNX vs sklearn predictions (first 5 rows):')
for i in range(5):
    print(f'  ONNX: row={onnx_pred[0][i][0]:.1f} col={onnx_pred[0][i][1]:.1f}  '
          f'sklearn: row={skl_pred[i][0]:.1f} col={skl_pred[i][1]:.1f}')
print('Verification complete.')